
# Native NeoOLAF DocRED batch run and evaluation — v6.1

This notebook freezes the successful **v5.1 Skai TV scientific configuration**
and applies it unchanged to a DocRED JSONL corpus.

The intended workflow is:

1. leave `RUN_ALL_DOCUMENTS = False` and run the first five documents;
2. inspect the aggregate and per-document evaluation;
3. change only `RUN_ALL_DOCUMENTS = True`;
4. rerun the batch cell to process every record in the JSONL.

Completed document runs are resumed. The first five documents are therefore not
paid for or executed again when the full-corpus run starts.

## Scientific constraints

- Native NeoOLAF Layers 0–12 run for every document.
- No file under `src/neoolaf` is modified.
- Gold entities and relations are removed before pipeline execution.
- The frozen v5.1 profile, user guidance, task guidance, ontology, and projection
  rules are identical for every document.
- No direct DocRED extractor, source-entity anchoring, gold-pair hints, or
  post-run relation invention is used.
- Gold data is loaded only after a document run for evaluation.
- Each document keeps its complete states, prompts, raw responses, logs,
  errors, ontology retrieval records, and analysis artifacts.
- Document-level launches use separate processes. Layer-level relation
  enrichment keeps its internal parallel workers.



## v6.1 recovery behavior

This patch fixes heterogeneous entity-projection CSV rows (for example, only
ambiguous rows contain `candidate_entity_ids`). It also detects a completed
Layer 0–12 `run_manifest.json` and rebuilds only the evaluation. Therefore,
the three documents that previously failed during CSV writing are recovered
without repeating any OpenRouter calls.


In [1]:

from __future__ import annotations

import json
import multiprocessing as mp
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_batch_v6_1 import (
    BatchRunConfig,
    aggregate_batch_results,
    iter_jsonl,
    read_json,
    run_batch,
)

mp.freeze_support()
print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Corpus path and the one-variable smoke/full switch

In [2]:

# Change only this variable after the five-document smoke run succeeds.
RUN_ALL_DOCUMENTS = False

# When RUN_ALL_DOCUMENTS=False, process exactly the first five matching records.
SMOKE_DOCUMENT_LIMIT = 5

# None means the entire JSONL regardless of its `type`/`split` field.
# Set to "dev" only when a split filter is scientifically intended.
TYPE_FILTER = None
START_INDEX = 0

# This is the full DocRED JSONL used by the earlier RAGTree notebooks.
DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# The same root is used for smoke and full runs. When RUN_ALL_DOCUMENTS becomes
# True, the first five completed document directories are detected and skipped.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_batch"

print("RUN_ALL_DOCUMENTS =", RUN_ALL_DOCUMENTS)
print("DATASET_JSONL =", DATASET_JSONL)
print("BATCH_ROOT =", BATCH_ROOT)


DATASET_JSONL=/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl
RUN_ALL_DOCUMENTS = False
DATASET_JSONL = /home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl
BATCH_ROOT = /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch


## 2. Frozen v5.1 scientific configuration and parallelism

In [3]:

ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Parallel native NeoOLAF launches. Keep this between 3 and 5 as requested.
DOCUMENT_WORKERS = 4

# Internal v5.1 Layer 2 workers retained per document.
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

# Safe large-run behavior.
RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1 if not RUN_ALL_DOCUMENTS else 10

# The maximum theoretical burst is DOCUMENT_WORKERS × LAYER_WORKERS.
# Lower DOCUMENT_WORKERS first if OpenRouter returns HTTP 429.
print("Document workers:", DOCUMENT_WORKERS)
print("Layer workers per document:", LAYER_WORKERS)
print("Maximum theoretical concurrent Layer 2 requests:", DOCUMENT_WORKERS * LAYER_WORKERS)
print("API key available:", bool(API_KEY))


Document workers: 4
Layer workers per document: 16
Maximum theoretical concurrent Layer 2 requests: 64
API key available: True


## 3. Preflight: files, corpus size, and anti-leakage assertions

In [4]:

required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

total_records = 0
matching_records = 0
first_ids = []
for record in iter_jsonl(DATASET_JSONL):
    total_records += 1
    actual_type = str(record.get("type") or record.get("split") or "")
    matches = TYPE_FILTER is None or TYPE_FILTER in {"", "all", "*"} or actual_type == TYPE_FILTER
    if matches:
        matching_records += 1
        if len(first_ids) < 5:
            first_ids.append(record.get("document_id") or record.get("id") or record.get("title"))

selected_count = matching_records if RUN_ALL_DOCUMENTS else min(SMOKE_DOCUMENT_LIMIT, matching_records)

print("JSONL records:", total_records)
print("Matching records:", matching_records)
print("Selected this invocation:", selected_count)
print("First selected IDs:", first_ids)
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False
assert 3 <= DOCUMENT_WORKERS <= 5

display(Markdown(
    f"**Mode:** {'full JSONL' if RUN_ALL_DOCUMENTS else 'five-document smoke'}  \\n"
    f"**Selected:** {selected_count} documents  \\n"
    f"**Resume:** {RESUME_COMPLETED}. The same batch root is retained across both modes."
))


JSONL records: 106924
Matching records: 106924
Selected this invocation: 5
First selected IDs: ['DocRED - e37288ca6012859f', 'DocRED - 72971f0f36693b3a', 'DocRED - 0af496a208c087f2', 'DocRED - 3b611c48af52425e', 'DocRED - 2d3e5aaa2e9e3e9b']
Ontology properties: 96
Allowed relation IDs: 96
Frozen profile: docred_native_country_compact_ablation_v5


**Mode:** five-document smoke  \n**Selected:** 5 documents  \n**Resume:** True. The same batch root is retained across both modes.


## 4. Run native NeoOLAF in parallel

Every document receives:

- one no-gold input JSONL;
- one separate gold JSONL used only after execution;
- a unique run directory;
- full Layer 0–12 checkpoints and artifacts;
- raw LLM responses and prompt audits;
- ontology-retrieval logs;
- per-document strict relation and entity evaluation.

The parent process writes `batch_events.jsonl` while documents finish. A failed
document does not terminate the other launches. Transient failures are retried
once by default.


In [5]:

RUN_PIPELINE = True

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    batch = run_batch(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=RUN_ALL_DOCUMENTS,
        smoke_document_limit=SMOKE_DOCUMENT_LIMIT,
        type_filter=TYPE_FILTER,
        start_index=START_INDEX,
        config=config,
        api_key=API_KEY,
    )
    print("Batch execution and aggregate evaluation completed.")
else:
    results = read_json(BATCH_ROOT / "document_results.json")
    batch = aggregate_batch_results(
        results=results,
        batch_root=BATCH_ROOT,
        relation_catalog_path=RELATION_CATALOG,
    )
    print("RUN_PIPELINE=False: aggregate evaluation rebuilt from saved results.")


/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/galencarmedeiro/git/postdoc/NeoOLAF/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


[1/5] completed: DocRED - e37288ca6012859f | pipeline=33.211s | reused saved pipeline
[2/5] completed: DocRED - 2d3e5aaa2e9e3e9b | pipeline=4.675s | reused saved pipeline
[3/5] completed: DocRED - 72971f0f36693b3a | pipeline=25.433s | reused saved pipeline
[4/5] completed: DocRED - 0af496a208c087f2 | pipeline=18.025s | reused saved pipeline
[5/5] completed: DocRED - 3b611c48af52425e | pipeline=3.952s | reused saved pipeline
Batch execution and aggregate evaluation completed.


## 5. Aggregate relation and entity evaluation

In [6]:

summary = batch["summary"]
display(pd.DataFrame([{
    "documents_requested": summary["documents_requested"],
    "documents_completed_or_resumed": summary["documents_completed_or_resumed"],
    "documents_failed": summary["documents_failed"],
    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],
    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],
    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],
    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],
    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}]))

print("First-failure counts across completed documents:")
display(pd.DataFrame(
    sorted(summary["first_failure_counts"].items(), key=lambda item: (-item[1], item[0])),
    columns=["first_failure", "gold_relations"],
))


,documents_requested,documents_completed_or_resumed,documents_failed,relation_micro_precision,relation_micro_recall,relation_micro_f1,relation_macro_precision,relation_macro_recall,relation_macro_f1,entity_micro_precision,entity_micro_recall,entity_micro_f1,endpoint_micro_precision,endpoint_micro_recall,endpoint_micro_f1,mean_pipeline_seconds,median_pipeline_seconds
0,5,5,0,0.59375,0.306452,0.404255,0.534286,0.36519,0.381115,1.0,0.638554,0.779412,0.826087,0.404255,0.542857,17.0592,18.025


First-failure counts across completed documents:


,first_failure,gold_relations
0,layer01_relation_instance_missing,30
1,survived_to_layer05,19
2,layer01_endpoint_missing_after_projection,13


## 6. Per-document results

In [7]:

per_document = pd.DataFrame(batch["per_document"])
if not per_document.empty:
    display(per_document.sort_values(
        ["relation_f1", "relation_recall", "relation_precision"],
        ascending=[True, True, True],
    ))
    display(per_document[[
        "relation_precision", "relation_recall", "relation_f1",
        "entity_precision", "entity_recall", "entity_f1",
        "pipeline_seconds",
    ]].describe())
else:
    print("No completed documents.")


,selection_index,source_index,document_id,title,status,pipeline_seconds,wall_seconds,relation_predicted,relation_gold,relation_tp,...,relation_precision,relation_recall,relation_f1,entity_precision,entity_recall,entity_f1,endpoint_precision,endpoint_recall,endpoint_f1,run_dir
4,4,4,DocRED - 2d3e5aaa2e9e3e9b,Conrad O. Johnson,completed,4.675,0.560,0,21,0,...,0.000000,0.000000,0.000000,1.0,0.695652,0.820513,0.000,0.000000,0.000000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
3,3,3,DocRED - 3b611c48af52425e,Lookin Ass,completed,3.952,0.512,1,6,1,...,1.000000,0.166667,0.285714,1.0,0.714286,0.833333,1.000,0.333333,0.500000,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
2,2,2,DocRED - 0af496a208c087f2,IBM Research – Brazil,completed,18.025,0.671,7,11,3,...,0.428571,0.272727,0.333333,1.0,0.312500,0.476190,1.000,0.555556,0.714286,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
1,1,1,DocRED - 72971f0f36693b3a,Washington Place (West Virginia),completed,25.433,0.996,14,17,9,...,0.642857,0.529412,0.580645,1.0,0.736842,0.848485,0.875,0.538462,0.666667,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...
0,0,0,DocRED - e37288ca6012859f,Skai TV,completed,33.211,0.756,10,7,6,...,0.600000,0.857143,0.705882,1.0,0.727273,0.842105,0.625,1.000000,0.769231,/home/galencarmedeiro/git/postdoc/NeoOLAF/exam...


,relation_precision,relation_recall,relation_f1,entity_precision,entity_recall,entity_f1,pipeline_seconds
count,5.000000,5.000000,5.000000,5.0,5.000000,5.000000,5.000000
mean,0.534286,0.365190,0.381115,1.0,0.637311,0.764125,17.059200
std,0.363879,0.335526,0.274736,0.0,0.182230,0.161302,12.816991
min,0.000000,0.000000,0.000000,1.0,0.312500,0.476190,3.952000
25%,0.428571,0.166667,0.285714,1.0,0.695652,0.820513,4.675000
50%,0.600000,0.272727,0.333333,1.0,0.714286,0.833333,18.025000
75%,0.642857,0.529412,0.580645,1.0,0.727273,0.842105,25.433000
max,1.000000,0.857143,0.705882,1.0,0.736842,0.848485,33.211000


## 7. Per-relation micro evaluation

In [8]:

per_relation = pd.DataFrame(batch["per_relation"])
if not per_relation.empty:
    display(per_relation.sort_values(
        ["gold", "f1", "recall"],
        ascending=[False, True, True],
    ))
else:
    print("No relation metrics available.")


,relation_id,label,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
5,P17,,16,21,12,4,9,0.750000,0.571429,0.648649
9,P27,,1,7,1,0,6,1.000000,0.142857,0.250000
3,P150,,0,5,0,0,5,0.000000,0.000000,0.000000
2,P131,,7,5,3,4,2,0.428571,0.600000,0.500000
21,P69,,0,3,0,0,3,0.000000,0.000000,0.000000
8,P264,,0,2,0,0,2,0.000000,0.000000,0.000000
10,P30,,0,2,0,0,2,0.000000,0.000000,0.000000
20,P577,,0,2,0,0,2,0.000000,0.000000,0.000000
22,P749,,0,2,0,0,2,0.000000,0.000000,0.000000
4,P159,,3,2,1,2,1,0.333333,0.500000,0.400000


## 8. Aggregate cumulative layer contribution

In [9]:

cumulative = pd.DataFrame(batch["cumulative"])
display(cumulative)

if not cumulative.empty:
    display(cumulative[[
        "layer_index", "layer_name", "predicted", "gold",
        "true_positive", "false_positive", "false_negative",
        "precision", "recall", "f1",
    ]])


,layer_index,layer_name,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,0,layer00_preprocessing,0,62,0,0,62,0.00000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,62,0,0,62,0.00000,0.000000,0.000000
2,2,layer02_candidate_enrichment,0,62,0,0,62,0.00000,0.000000,0.000000
3,3,layer03_candidate_typing_resolution,0,62,0,0,62,0.00000,0.000000,0.000000
4,4,layer04_candidate_relation_extraction,32,62,19,13,43,0.59375,0.306452,0.404255
5,5,layer05_candidate_triple_generation,32,62,19,13,43,0.59375,0.306452,0.404255
6,6,layer06_concept_relation_induction,32,62,19,13,43,0.59375,0.306452,0.404255
7,7,layer07_hierarchisation,32,62,19,13,43,0.59375,0.306452,0.404255
8,8,layer08_axiom_schemata_extraction,32,62,19,13,43,0.59375,0.306452,0.404255
9,9,layer09_general_axiom_extraction,32,62,19,13,43,0.59375,0.306452,0.404255


,layer_index,layer_name,predicted,gold,true_positive,false_positive,false_negative,precision,recall,f1
0,0,layer00_preprocessing,0,62,0,0,62,0.00000,0.000000,0.000000
1,1,layer01_linguistic_expression_extraction,0,62,0,0,62,0.00000,0.000000,0.000000
2,2,layer02_candidate_enrichment,0,62,0,0,62,0.00000,0.000000,0.000000
3,3,layer03_candidate_typing_resolution,0,62,0,0,62,0.00000,0.000000,0.000000
4,4,layer04_candidate_relation_extraction,32,62,19,13,43,0.59375,0.306452,0.404255
5,5,layer05_candidate_triple_generation,32,62,19,13,43,0.59375,0.306452,0.404255
6,6,layer06_concept_relation_induction,32,62,19,13,43,0.59375,0.306452,0.404255
7,7,layer07_hierarchisation,32,62,19,13,43,0.59375,0.306452,0.404255
8,8,layer08_axiom_schemata_extraction,32,62,19,13,43,0.59375,0.306452,0.404255
9,9,layer09_general_axiom_extraction,32,62,19,13,43,0.59375,0.306452,0.404255


## 9. Failures, resume state, and saved outputs

In [10]:

failed = pd.DataFrame(batch["failed"])
print("Failed documents:", len(failed))
if not failed.empty:
    display(failed[[
        column for column in [
            "selection_index", "document_id", "title", "status",
            "error_type", "error", "run_dir",
        ]
        if column in failed.columns
    ]])

print("Batch output root:", BATCH_ROOT)
print("Aggregate summary:", BATCH_ROOT / "aggregate_analysis/batch_summary.json")
print("Per-document CSV:", BATCH_ROOT / "aggregate_analysis/per_document_metrics.csv")
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv")
print("Predictions JSONL:", BATCH_ROOT / "aggregate_analysis/predictions.jsonl")
print("Failures JSONL:", BATCH_ROOT / "aggregate_analysis/failed_documents.jsonl")
print("Progress events:", BATCH_ROOT / "batch_events.jsonl")
print("Selection manifest:", BATCH_ROOT / "selection_manifest.json")


Failed documents: 0
Batch output root: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch
Aggregate summary: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch/aggregate_analysis/batch_summary.json
Per-document CSV: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch/aggregate_analysis/per_document_metrics.csv
Per-relation CSV: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch/aggregate_analysis/per_relation_metrics.csv
Cumulative layer CSV: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch/aggregate_analysis/cumulative_layer_micro_evaluation.csv
Predictions JSONL: /home/galencarmedeiro/git/postdoc/NeoOLAF/examples/RAGTreeDatasets/runs/docred_native_v5_1_batch/aggregate_analysis/predictions.jsonl
Failures JSONL: /home/galencarmedeiro/git/postdoc/NeoOLAF/examp


## 10. Full-corpus launch

After the five-document result is acceptable:

```python
RUN_ALL_DOCUMENTS = True
```

Rerun the configuration, preflight, and batch cells. Do not change the batch
root. The first five scientific fingerprints match and their completed runs are
loaded rather than executed again.

For HTTP 429 errors, reduce `DOCUMENT_WORKERS` from 4 to 3 before reducing the
internal `LAYER_WORKERS`. This preserves most of the per-document v5.1 speedup.
